# NB1 — Multi-Provider LLM API Calls
## Session 05: Ollama & Beyond — Calling 6 LLM Providers with One Pattern

---

### What you'll learn
By the end of this notebook you will be able to:
- Understand the **universal message format** that all modern LLM APIs share
- Call **6 different providers** — Ollama, LM Studio, OpenAI, Anthropic, Gemini, and OpenRouter
- Build a **single `call_llm()` function** that works with any provider
- Compare responses from multiple models on the **same question**

---

### ⚠️ Local vs Cloud — Read This First

| Provider | Where it runs | Cost | Requires |
|---|---|---|---|
| **Ollama** | Your laptop | Free | Ollama installed + model pulled |
| **LM Studio** | Your laptop | Free | LM Studio installed + model loaded |
| **OpenAI** | Cloud | Paid | API key from platform.openai.com |
| **Anthropic** | Cloud | Paid | API key from console.anthropic.com |
| **Gemini** | Cloud | Free tier ✅ | API key from aistudio.google.com |
| **OpenRouter** | Cloud | Free models ✅ | API key from openrouter.ai |

> **Running in Google Colab?** Skip Ollama and LM Studio sections — they need a local server.  
> Use **Gemini** (free) and **OpenRouter** (free models) instead.  
> Both work perfectly in Colab with zero setup beyond getting an API key.

---

### The Big Idea
All LLM providers — no matter how different they look — follow the **same core pattern**:
```
You send:  [ { role: "system", content: "..." }, { role: "user", content: "..." } ]
You get:   { choices: [ { message: { content: "..." } } ] }
```
Once you understand this pattern, switching providers is trivial. Let's prove it.

In [ ]:
# Install all required libraries
# openai      → used for OpenAI, Ollama, LM Studio, and OpenRouter (they all speak the same protocol)
# anthropic   → Claude-specific SDK (slightly different API shape)
# google-genai        → Gemini SDK (new unified SDK)
# python-dotenv → load API keys from a .env file
# requests    → for the raw HTTP demo in Section 1
!pip install openai anthropic google-genai python-dotenv requests -q

---

## SECTION 1 — CONCEPT: How LLM APIs Work

### The Universal Message Pattern

Think of calling an LLM API like sending a text message to a very smart assistant:
- You write a **system message** (sets the assistant's personality/role)
- You write a **user message** (your actual question)
- The assistant writes back in the **assistant message**

Here's what the data flow looks like:

```
YOUR CODE                    NETWORK                   LLM SERVER
─────────────────────────────────────────────────────────────────────

  messages = [           ──────── HTTP POST ──────►   Model processes
    {                                                  your messages
      role: "system",    ◄──── JSON response ──────   and generates
      content: "...",                                  a reply
    },
    {
      role: "user",
      content: "...",     The response contains:
    }                      choices[0].message.content
  ]                        = the actual text reply
```

### Why does every provider look the same?
OpenAI published this message format first. It became so popular that **everyone else copied it** — Ollama, LM Studio, OpenRouter all serve an "OpenAI-compatible" API. This is great news: once you learn one, you know them all.

The only exception is **Anthropic** (Claude) which has a slightly different shape — we'll see that difference in detail below.

### Let's start at the lowest level: raw HTTP

In [ ]:
import requests  # standard HTTP library — sends web requests
import json      # for pretty-printing JSON responses

# ⚠️ OLLAMA MUST BE RUNNING LOCALLY FOR THIS CELL
# If you don't have Ollama, skip to Cell 5 (Provider 1) and come back later.

# 👇 This is the RAW HTTP call — every SDK (openai, anthropic, etc.)
#    does exactly this under the hood, just with nicer Python syntax.
response = requests.post(
    "http://localhost:11434/v1/chat/completions",  # Ollama's API endpoint
    headers={"Content-Type": "application/json"},  # tell the server we're sending JSON
    json={
        "model": "llama3.1:latest",         # which model to use (must be pulled already)
        "messages": [                   # the conversation history
            {
                "role": "system",       # sets the model's behaviour
                "content": "You are a helpful assistant."
            },
            {
                "role": "user",         # our question
                "content": "What is the capital of France? One sentence only."
            }
        ],
        "temperature": 0.7,             # 0=focused/deterministic, 1=creative/random
        "max_tokens": 100,              # hard cap on response length
    }
)

# 👇 The response is JSON — let's parse it and grab just the text
data = response.json()

# Show the full structure so you can see where the text lives
print("=== Full JSON response (abbreviated) ===")
print(json.dumps(data, indent=2)[:800])   # first 800 chars so it doesn't flood the screen

print("\n=== Just the answer ===")
print(data["choices"][0]["message"]["content"])  # navigate the nested dict

---

## Provider 1: Ollama (Local, Free)

### What is Ollama?
Ollama is a tool that lets you run open-source LLMs (like Qwen, Llama, Mistral) **directly on your laptop** — no internet needed, no API key, completely free.

**Think of it like:** Netflix vs buying a DVD. Normally you stream from the cloud (API). With Ollama, you have the movie stored locally.

### Setup (one-time)
```bash
# 1. Install Ollama from https://ollama.com
# 2. Pull a model (this downloads it to your laptop)
ollama pull qwen2.5:7b
# 3. Ollama starts automatically as a background service
```

### The clever trick
Ollama **mimics the OpenAI API** at `http://localhost:11434/v1`.  
This means you can use the OpenAI Python SDK to talk to Ollama — just change `base_url`.

⚠️ **Colab users:** This section requires Ollama running on your local machine. Skip ahead to Provider 5 (Gemini) if you're on Colab.

In [ ]:
from openai import OpenAI  # the official OpenAI Python SDK

# 👇 Key insight: we're using the OpenAI SDK but pointing it at OLLAMA
#    by changing base_url to localhost. The SDK doesn't know or care
#    that it's talking to a local model instead of OpenAI's servers.
ollama_client = OpenAI(
    base_url="http://localhost:11434/v1",  # ⚠️ Local only — Ollama's address
    api_key="ollama",                       # Ollama ignores this, but the SDK requires a value
)

response = ollama_client.chat.completions.create(
    model="llama3.1:latest",      # model name must match what you pulled with `ollama pull`
    messages=[
        {
            "role": "system",
            "content": "You are a helpful AI tutor. Be concise."
        },
        {
            "role": "user",
            "content": "Explain what an embedding is in 2 sentences."
        },
    ],
    temperature=0.7,
    max_tokens=200,
)

# 👇 The response object is identical to what you'd get from real OpenAI
print(response.choices[0].message.content)
print(f"\nTokens used: {response.usage.total_tokens}")

In [ ]:
# 👇 STREAMING: tokens appear word-by-word as the model generates them
#    Without streaming, you wait for the entire response before seeing anything.
#    With streaming, you see text appear like someone is typing in real time.

stream = ollama_client.chat.completions.create(
    model="llama3.1:latest",
    messages=[
        {"role": "user", "content": "Count from 1 to 10 slowly, one number per line."}
    ],
    stream=True,    # 👈 This single flag enables streaming — that's all it takes
)

print("Streaming response (watch tokens appear):", flush=True)
print("-" * 40)

for chunk in stream:          # each iteration gives us a tiny piece of the response
    delta = chunk.choices[0].delta.content   # the new text in this chunk
    if delta:                 # some chunks are empty (metadata only), skip those
        print(delta, end="", flush=True)  # print without newline, flush immediately

print()   # final newline to close the output cleanly

---

## Provider 2: LM Studio (Local, Free)

### What is LM Studio?
LM Studio is a **desktop app** that gives you a GUI for running local LLMs. It's like Ollama but with a visual interface — great for people who prefer not to use the terminal.

**The API it serves is identical to Ollama** — the only difference is the port number.

### Setup (one-time)
1. Download from [lmstudio.ai](https://lmstudio.ai)
2. Search and download a model inside the app (e.g. `qwen3-4b`)
3. Go to the **Local Server** tab and press **Start Server**
4. The server runs at `http://localhost:1234`

⚠️ **Colab users:** Same as Ollama — requires a local running app. Skip to Provider 5 (Gemini).

In [ ]:
# 👇 LM Studio serves the same OpenAI-compatible API as Ollama
#    The ONLY difference from the Ollama client above is:
#    - Different port: 1234 instead of 11434
#    - Different model name (must match what you loaded in LM Studio)

lmstudio_client = OpenAI(
    base_url="http://localhost:1234/v1",   # ⚠️ LM Studio must be running with server started
    api_key="lm-studio",                    # LM Studio ignores this, but the SDK needs a value
)

response = lmstudio_client.chat.completions.create(
    model="qwen/qwen3-4b-2507",   # 👈 This must match the model you loaded in LM Studio
    messages=[
        {
            "role": "user",
            "content": "What is RAG in AI? Explain in 2 sentences."
        },
    ],
    temperature=0.5,   # slightly lower temperature for a more precise answer
    max_tokens=200,
)

print(response.choices[0].message.content)
print(f"\nModel used: {response.model}")

---

## Provider 3: OpenAI (Cloud, Paid)

### What is OpenAI's API?
This is the original — the API that defined the message format everyone else copied. OpenAI hosts GPT-4, GPT-4o, and other models on their cloud servers.

**Think of it like:** Renting a supercomputer by the millisecond. You pay only for what you use.

### Getting an API key
1. Go to [platform.openai.com](https://platform.openai.com)
2. Create an account → API Keys → Create new key
3. Add a small amount of credit ($5 goes a long way with `gpt-4o-mini`)

### Model choice: `gpt-4o-mini`
For learning purposes, always use `gpt-4o-mini`. It's:
- GPT-4 class quality (not a dumbed-down model)
- ~30x cheaper than `gpt-4o`
- Fast response times

In [ ]:
import os

# 👇 Two ways to set your API key:
#    Option A: paste it directly here (easiest for learning, not safe for production)
#    Option B: set environment variable OPENAI_API_KEY and read it with os.getenv()

OPENAI_API_KEY = "sk-..."   # 👈 Replace with your real key from platform.openai.com
# OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")  # Option B: safer for production

# 👇 No base_url needed — OpenAI() defaults to api.openai.com
openai_client = OpenAI(api_key=OPENAI_API_KEY)

response = openai_client.chat.completions.create(
    model="gpt-4o-mini",          # cheapest GPT-4 class model
    messages=[
        {
            "role": "system",
            "content": "You are a concise AI tutor. Never use more than 3 sentences."
        },
        {
            "role": "user",
            "content": "What is the difference between fine-tuning and RAG?"
        },
    ],
    temperature=0.7,
    max_tokens=300,
)

print(response.choices[0].message.content)

# 👇 Calculate approximate cost
# gpt-4o-mini pricing: ~$0.15 per million input tokens, $0.60 per million output tokens
input_cost  = response.usage.prompt_tokens     * 0.00000015   # $0.15 per 1M tokens
output_cost = response.usage.completion_tokens * 0.00000060   # $0.60 per 1M tokens
print(f"\nTokens: {response.usage.prompt_tokens} in + {response.usage.completion_tokens} out")
print(f"Cost estimate: ~${input_cost + output_cost:.6f}")

---

## Provider 4: Anthropic / Claude (Cloud, Paid)

### What makes Anthropic different?
Anthropic makes the Claude family of models. The API has a **slightly different shape** from OpenAI:

| Feature | OpenAI | Anthropic |
|---|---|---|
| System prompt | Inside `messages` list | Separate `system=` parameter |
| Response text | `response.choices[0].message.content` | `response.content[0].text` |
| Token counts | `response.usage.total_tokens` | `response.usage.input_tokens` + `output_tokens` |
| SDK | `openai` | `anthropic` |

### Getting an API key
1. Go to [console.anthropic.com](https://console.anthropic.com)
2. Create an account → API Keys → Create Key

### Model choice: `claude-haiku-4-5-20251001`
Haiku is Anthropic's fastest and cheapest model — perfect for learning and experimentation.

In [ ]:
import anthropic  # Anthropic's own SDK — different from openai

ANTHROPIC_API_KEY = "sk-ant-..."   # 👈 Replace with your key from console.anthropic.com

# 👇 Anthropic has its own client class
ant_client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

# 👇 Notice the difference from OpenAI:
#    - system= is a SEPARATE parameter, not inside messages[]
#    - max_tokens is REQUIRED (not optional)
#    - the method is .messages.create() (same concept, slightly different name)
response = ant_client.messages.create(
    model="claude-haiku-4-5-20251001",   # fastest and cheapest Claude model
    max_tokens=300,                       # required for Anthropic (optional for OpenAI)
    system="You are a concise AI tutor. Never use more than 3 sentences.",
    messages=[
        {
            "role": "user",
            "content": "What makes attention mechanisms special?"
        },
    ],
)

# 👇 Response shape is different: response.content[0].text  (not choices[0].message.content)
print(response.content[0].text)

# Anthropic separates input and output tokens instead of combining them
print(f"\nTokens: {response.usage.input_tokens} in, {response.usage.output_tokens} out")

---

## Provider 5: Google Gemini (Cloud, Free Tier Available ✅)

### What is Gemini?
Gemini is Google's family of LLMs. `gemini-2.0-flash` is fast, capable, and available on a **generous free tier** — perfect for this course.

**Think of it like:** Google's version of ChatGPT, but with a free API you can use in code.

### Getting an API key (Free!)
1. Go to [aistudio.google.com](https://aistudio.google.com)
2. Sign in with Google → click **Get API Key**
3. No credit card needed for the free tier

### Gemini's API shape
Gemini has its own SDK (`google-generativeai`) with a different structure again:
- You create a model object first: `genai.GenerativeModel("model-name")`
- Then call `.generate_content("your prompt")` on it
- Response text lives at `response.text`

✅ **Colab users:** This section works perfectly in Colab!

In [ ]:
from google import genai            # Google's new unified Gemini SDK

GOOGLE_API_KEY = "AIza..."   # 👈 Free key from aistudio.google.com — no credit card needed!

# 👇 Step 1: create a client with your API key
client_gemini = genai.Client(api_key=GOOGLE_API_KEY)

# 👇 Step 2: generate content — pass model and prompt directly to the client
#    Gemini takes a plain string as 'contents', unlike OpenAI's messages list
response = client_gemini.models.generate_content(
    model="gemini-2.5-flash",
    contents="Explain transformer attention in 3 bullet points.",
    config=genai.types.GenerateContentConfig(
        temperature=0.7,          # same meaning as OpenAI's temperature
        max_output_tokens=300,    # note: max_output_tokens, not max_tokens
    ),
)

# 👇 Response text is at response.text — much simpler than OpenAI's nested structure
print(response.text)


---

## Provider 6: OpenRouter (Cloud, Free Models Available ✅)

### What is OpenRouter?
OpenRouter is a **gateway to 200+ models** from one API key. It supports models from:
- Meta (Llama)
- Mistral
- Google (Gemma)
- Microsoft (Phi)
- And many more

**The killer feature:** Many models are completely **free** (marked with `:free` suffix).

**Think of it like:** A universal remote control that works with every TV brand. One API key, any model.

### Getting an API key (Free!)
1. Go to [openrouter.ai](https://openrouter.ai)
2. Sign up → Keys → Create Key
3. Free models are available immediately — no payment needed

### The API shape
OpenRouter is **fully OpenAI-compatible** — just change `base_url`. This is the same trick we used with Ollama and LM Studio.

✅ **Colab users:** This works perfectly in Colab! Use the `:free` suffix models.

In [ ]:
# 👇 OpenRouter = OpenAI SDK + different base_url + OpenRouter API key
#    Same pattern as Ollama and LM Studio, but this time the server is in the cloud

OPENROUTER_API_KEY = "sk-or-..."   # 👈 Free key from openrouter.ai

openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",   # OpenRouter's cloud endpoint
    api_key=OPENROUTER_API_KEY,
)

# 👇 ":free" at the end of the model name means it costs nothing
#    You get a 70-billion parameter model for free — remarkable!
response = openrouter_client.chat.completions.create(
    model="meta-llama/llama-3.3-70b-instruct:free",   # 70B parameter model, completely free
    messages=[
        {
            "role": "user",
            "content": "What is quantization in LLMs? Explain in 2-3 sentences."
        },
    ],
    max_tokens=300,
)

# 👇 Same response shape as OpenAI — identical code, different provider
print(response.choices[0].message.content)
print(f"\nModel: {response.model}")

---

## SECTION 2 — GUIDED BUILD: Universal LLM Caller

### The Goal
You've now seen 6 different providers. Notice how similar they all look? Let's take that observation and build a **single function** that handles all of them.

**Why is this useful?**
- Switch providers by changing one string: `call_llm("gemini", ...)` → `call_llm("openai", ...)`
- Build applications that can use any provider without rewriting logic
- This is the real-world pattern used in production LLM apps

### Your task
Fill in the `___` placeholders. Each one is labelled with a `# TODO:` comment telling you exactly what to put there.

**Hint:** Look at the code you've already written in Section 1 — the answers are all there!

In [ ]:
# 👇 UNIVERSAL LLM CALLER
#    One function that works with any of the 6 providers.
#    Fill in the ___ parts using what you learned in Section 1.

def call_llm(provider: str, prompt: str, api_key: str = "", model: str = "") -> str:
    """
    Call any LLM provider with the same interface.

    Args:
        provider : One of: "ollama" | "lmstudio" | "openai" | "anthropic" | "gemini" | "openrouter"
        prompt   : The question or instruction to send to the model
        api_key  : Your API key (leave empty for local providers Ollama and LM Studio)
        model    : Model name — if empty, a sensible default is used for each provider

    Returns:
        The model's response as a plain Python string
    """

    # ------------------------------------------------------------------ #
    #  OLLAMA — local server at port 11434                                 #
    # ------------------------------------------------------------------ #
    if provider == "ollama":
        client = OpenAI(
            base_url=___,       # TODO: fill in the Ollama base URL (e.g. "http://localhost:11434/v1")
            api_key="ollama",   # Ollama ignores the key — any string works
        )
        model = model or "llama3.1:latest"   # default model if caller didn't specify one
        resp = client.chat.completions.create(
            model=___,                                      # TODO: use the model variable
            messages=[{"role": "user", "content": ___}],   # TODO: use the prompt variable
            max_tokens=500,
        )
        return resp.choices[0].message.content   # extract the text from the response

    # ------------------------------------------------------------------ #
    #  LM STUDIO — local server at port 1234                              #
    # ------------------------------------------------------------------ #
    elif provider == "lmstudio":
        client = OpenAI(
            base_url=___,           # TODO: fill in the LM Studio base URL (e.g. "http://localhost:1234/v1")
            api_key="lm-studio",   # LM Studio ignores the key — any string works
        )
        model = model or "qwen/qwen3-4b-2507"   # default — must match model loaded in LM Studio
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
        )
        return ___   # TODO: extract and return the text from resp (same pattern as Ollama above)

    # ------------------------------------------------------------------ #
    #  OPENAI — cloud API at api.openai.com                               #
    # ------------------------------------------------------------------ #
    elif provider == "openai":
        client = OpenAI(api_key=___)   # TODO: pass the api_key parameter to the client
        model = model or "gpt-4o-mini"  # cheapest GPT-4 class model
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
        )
        return resp.choices[0].message.content

    # ------------------------------------------------------------------ #
    #  ANTHROPIC — cloud API, different SDK shape                         #
    # ------------------------------------------------------------------ #
    elif provider == "anthropic":
        import anthropic as ant
        client = ant.Anthropic(api_key=___)   # TODO: pass the api_key parameter
        model = model or "claude-haiku-4-5-20251001"   # fastest, cheapest Claude
        resp = client.messages.create(
            model=model,
            max_tokens=500,
            messages=[{"role": "user", "content": ___}],  # TODO: use the prompt variable
        )
        return resp.content[0].text   # Anthropic uses .content[0].text, not .choices[0].message.content

    # ------------------------------------------------------------------ #
    #  GEMINI — Google's cloud API, its own SDK                           #
    # ------------------------------------------------------------------ #
    elif provider == "gemini":
        import google.generativeai as genai
        genai.configure(api_key=___)   # TODO: configure with the api_key parameter
        model = model or "gemini-2.0-flash"   # fast and free-tier friendly
        m = genai.GenerativeModel(___)         # TODO: pass the model variable here
        return m.generate_content(prompt).text  # .text is Gemini's equivalent of .choices[0].message.content

    # ------------------------------------------------------------------ #
    #  OPENROUTER — cloud gateway to 200+ models                          #
    # ------------------------------------------------------------------ #
    elif provider == "openrouter":
        client = OpenAI(
            base_url=___,    # TODO: OpenRouter's base URL (e.g. "https://openrouter.ai/api/v1")
            api_key=api_key,
        )
        model = model or "meta-llama/llama-3.3-70b-instruct:free"   # free 70B model
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            max_tokens=500,
        )
        return ___   # TODO: return the response text (same pattern as OpenAI)

    # ------------------------------------------------------------------ #
    #  Unknown provider — fail clearly with a helpful message             #
    # ------------------------------------------------------------------ #
    else:
        raise ValueError(
            f"Unknown provider: '{provider}'. "
            f"Choose from: ollama, lmstudio, openai, anthropic, gemini, openrouter"
        )


# 👇 Quick smoke test — run this to verify Ollama works
#    (swap "ollama" for "gemini" or "openrouter" if you're on Colab)
result = call_llm("ollama", "What is 2 + 2? Answer with one word only.")
print(f"Test passed! Ollama says: {result}")

---

## SECTION 3 — EXPERIMENT: Compare Responses Across Providers

### What we're testing
Now that `call_llm()` works with any provider, let's use it to run the **same question** across multiple providers and compare:
1. The **quality** of answers — are they accurate? Complete?
2. The **speed** — how long does each provider take?
3. The **style** — do different models have different personalities?

### Configure your providers below
- Uncomment the providers you have keys for
- Ollama/LM Studio work without keys (if running locally)
- Gemini and OpenRouter have free tiers — highly recommended

In [ ]:
import time   # for measuring how long each provider takes to respond

# 👇 The SAME question will be sent to every provider you configure below
QUESTION = "What is the most important thing to understand about large language models?"

# 👇 Configure which providers to test.
#    Each entry is: "provider_name": {"api_key": "...", "model": "..."}
#    Set model to "" to use the default for that provider.
#    Uncomment lines as you add API keys.

MY_PROVIDERS = {
    "ollama": {"api_key": "", "model": "llama3.1:latest"},   # Local — no key needed
    # "lmstudio":   {"api_key": "",            "model": ""},
    # "openai":     {"api_key": "sk-...",       "model": "gpt-4o-mini"},
    # "anthropic":  {"api_key": "sk-ant-...",   "model": ""},
    # "gemini":     {"api_key": "AIza...",       "model": "gemini-2.0-flash"},
    # "openrouter": {"api_key": "sk-or-...",    "model": "meta-llama/llama-3.3-70b-instruct:free"},
}

print(f"Question: {QUESTION}")
print("=" * 60)

for provider_name, config in MY_PROVIDERS.items():
    print(f"\n{'─' * 40}")
    print(f"Provider: {provider_name.upper()}")
    print(f"Model:    {config['model'] or '(default)'}")
    print(f"{'─' * 40}")

    start_time = time.time()   # record when we started the request
    try:
        # 👇 call_llm() hides all the provider differences — same call for every provider
        answer = call_llm(provider_name, QUESTION, **config)
        elapsed = round(time.time() - start_time, 2)   # calculate how long it took

        print(answer)
        print(f"\nTime: {elapsed}s")
    except Exception as e:
        # 👇 If a provider fails (wrong key, server not running, etc.) show the error
        #    and continue to the next provider instead of crashing the whole cell
        print(f"Error: {e}")

In [ ]:
# ─────────────────────────────────────────────────────────────
#  EXPERIMENT: How does TEMPERATURE change the output?
# ─────────────────────────────────────────────────────────────

# Temperature controls how "random" or "creative" the model is:
#   0.0 = very focused, picks the most likely next token every time (deterministic)
#   0.5 = balanced — some creativity, still mostly coherent
#   1.0 = very creative, sometimes surprising, occasionally weird
#   >1.0 = chaotic (usually not useful)

# 👇 But wait — call_llm() doesn't have a temperature parameter yet!
#    This is intentional. For this experiment we call the API directly
#    so we can pass temperature. You'll add it to call_llm() in Section 4.

PROMPT = "Describe artificial intelligence in one sentence. Be creative."

print("Temperature experiment — same prompt, 3 different temperature values:")
print(f"Prompt: '{PROMPT}'\n")

# 👇 We'll use Ollama directly here (change to another provider if needed)
exp_client = OpenAI(
    base_url="http://localhost:11434/v1",
    api_key="ollama",
)

for temperature in [0.0, 0.5, 1.0]:
    resp = exp_client.chat.completions.create(
        model="llama3.1:latest",
        messages=[{"role": "user", "content": PROMPT}],
        temperature=temperature,   # 👈 the variable we're experimenting with
        max_tokens=100,
    )
    answer = resp.choices[0].message.content.strip()
    print(f"Temperature {temperature}: {answer}")
    print()   # blank line between results for readability

# 👇 What to observe:
#    - Temperature 0.0 should give the same (or very similar) answer if you run it twice
#    - Temperature 1.0 should give noticeably different answers on each run
#    - Try running this cell multiple times and compare!

In [ ]:
# ─────────────────────────────────────────────────────────────
#  EXPERIMENT: How does max_tokens affect the response?
# ─────────────────────────────────────────────────────────────

# max_tokens is a hard cutoff — the model stops generating after this many tokens.
# 1 token ≈ 0.75 words in English.
# Setting it too low can cut off mid-sentence — sometimes comically!

PROMPT_LONG = "Explain how neural networks learn. Be thorough and detailed."

print("max_tokens experiment — same prompt, 3 different length limits:\n")

for max_tok in [20, 100, 500]:
    resp = exp_client.chat.completions.create(
        model="llama3.1:latest",
        messages=[{"role": "user", "content": PROMPT_LONG}],
        temperature=0.5,
        max_tokens=max_tok,    # 👈 the variable we're experimenting with
    )
    answer = resp.choices[0].message.content.strip()
    actual_tokens = resp.usage.completion_tokens   # how many tokens were actually generated
    print(f"--- max_tokens={max_tok} (generated {actual_tokens}) ---")
    print(answer)
    print()

---

## SECTION 4 — CHALLENGE

You've built the foundation. Now extend it.

These challenges push you to think like a **developer building real LLM-powered tools**, not just a user making API calls.

---

### Challenge 1: Add Streaming Support to `call_llm()`

The current `call_llm()` function returns the full response only after the model finishes generating. In a real app, you want to **stream** the output so the user sees text appear in real time.

**Your task:** Add a `stream: bool = False` parameter to `call_llm()`.  
When `stream=True`:
- Print each token as it arrives (for local providers that support it)
- Return the full assembled text at the end
- For providers that don't support streaming (Gemini), return the full text normally

**Hints:**
- You've already seen streaming in Section 1 (Cell 7) — the pattern is `for chunk in stream: print(chunk.choices[0].delta.content)`
- Use a `parts = []` list, append each chunk, then `return "".join(parts)` at the end
- Gemini's `generate_content` doesn't stream — just call it normally when `stream=True` for Gemini
- The `flush=True` argument in `print()` ensures text appears immediately, not buffered

In [ ]:
# CHALLENGE 1: Streaming version of call_llm()
# Add stream=False as a new parameter and implement streaming for Ollama + LM Studio + OpenAI + OpenRouter

def call_llm_streaming(provider: str, prompt: str, api_key: str = "", model: str = "", stream: bool = False) -> str:
    """
    Extended version of call_llm() with optional streaming support.
    When stream=True: print tokens as they arrive, then return the full text.
    When stream=False: behave exactly like the original call_llm().
    """
    # YOUR CODE HERE
    # Hint: copy the relevant provider blocks from call_llm() and modify them
    # Hint: for streaming providers, set stream=True in .create() and iterate chunks
    pass


# Test when you're ready:
# result = call_llm_streaming("ollama", "Count from 1 to 5.", stream=True)
# print(f"\nFull result: {result}")

---

### Challenge 2: Build `call_all_providers()`

Build a function that calls **all configured providers simultaneously** (using threads for speed) and returns a dictionary mapping provider names to their responses.

**Your task:** Write `call_all_providers(prompt, providers_config)` that:
- Takes a prompt and the same `MY_PROVIDERS` dict format used in Section 3
- Calls each provider **in parallel** using `concurrent.futures.ThreadPoolExecutor`
- Returns a dict like `{"ollama": "...", "gemini": "...", ...}`
- Handles errors gracefully — if one provider fails, include `{"error": "..."}` for that key
- Prints a summary showing which providers succeeded and how long each took

**Hints:**
- `from concurrent.futures import ThreadPoolExecutor, as_completed`
- `executor.submit(call_llm, provider, prompt, **config)` submits a task and returns a Future
- `future.result()` gets the return value when the task completes
- Wrap `future.result()` in try/except to catch provider errors without crashing

In [ ]:
# CHALLENGE 2: Parallel multi-provider caller
from concurrent.futures import ThreadPoolExecutor, as_completed

def call_all_providers(prompt: str, providers_config: dict) -> dict:
    """
    Call all providers in parallel and return a dict of results.

    Args:
        prompt           : The question to ask all providers
        providers_config : Dict like MY_PROVIDERS — {provider_name: {api_key, model}}

    Returns:
        Dict mapping provider name → response string (or {"error": ...} on failure)
    """
    # YOUR CODE HERE
    # Hint: use ThreadPoolExecutor with max_workers=len(providers_config)
    # Hint: keep a dict {future: provider_name} so you can match results back to providers
    pass


# Test when you're ready:
# results = call_all_providers("What is 1 + 1?", MY_PROVIDERS)
# for provider, answer in results.items():
#     print(f"{provider}: {answer}")

---

### Challenge 3: Build `compare_table()`

Take the results dict from `call_all_providers()` and display it as a nicely formatted comparison table.

**Your task:** Write `compare_table(results, question)` that:
- Prints a markdown-style table with columns: Provider | Response (first 150 chars) | Word Count | Status
- Status should be `OK` if it's a real response and `ERROR` if the value contains `{"error": ...}`
- Word count is just `len(response.split())`
- Also prints which provider gave the **longest** and **shortest** answer

**Extended challenge:** Use `pandas` to create the table and display it in the notebook:
```python
import pandas as pd
df = pd.DataFrame(rows)   # rows is a list of dicts
display(df)               # renders as a formatted table in Jupyter
```

**Hints:**
- `response[:150] + "..."` truncates a string to the first 150 characters
- `max(results.items(), key=lambda x: len(x[1].split()))` finds the provider with most words
- Handle the `{"error": ...}` case — check if the string starts with `"Error"` or if it's a dict

In [ ]:
# CHALLENGE 3: Comparison table printer
import pandas as pd   # for creating a nicely formatted table

def compare_table(results: dict, question: str) -> None:
    """
    Display a comparison table of responses from multiple providers.

    Args:
        results  : Dict from call_all_providers() — {provider_name: response_string}
        question : The original prompt (printed at the top for context)
    """
    # YOUR CODE HERE
    # Hint: build a list of dicts like:
    #   [{"Provider": "ollama", "Preview": "...", "Words": 42, "Status": "OK"}, ...]
    # Hint: use pandas DataFrame to display it cleanly
    # Hint: print a summary line at the bottom: "Longest: X | Shortest: Y"
    pass


# Test when you're ready (after completing Challenges 1 and 2):
# results = call_all_providers("Explain AI in one sentence.", MY_PROVIDERS)
# compare_table(results, "Explain AI in one sentence.")

---

## Summary — What You've Learned

| Concept | Key Takeaway |
|---|---|
| Universal message format | All LLMs accept `[{role, content}, ...]` — learn once, use everywhere |
| OpenAI-compatible APIs | Ollama, LM Studio, OpenRouter all speak the same protocol |
| Anthropic's difference | `system=` is a separate parameter; `.content[0].text` not `.choices[0].message.content` |
| Gemini's difference | Different SDK, but `.text` is even simpler than OpenAI |
| Temperature | 0 = focused/deterministic, 1 = creative/random |
| max_tokens | Hard cutoff — set too low and responses get cut off mid-sentence |
| Streaming | Just add `stream=True` and iterate chunks — works on most providers |
| Provider abstraction | Wrapping providers in one function makes your app portable |

---

### What's MISSING? →

We can now call 6 different providers. But every call is a **single question → single answer**.  
Real applications need **memory** — the ability to have a conversation that builds on previous messages.

**Next session:** We'll build a chatbot with conversation history, sliding window memory, and session persistence — and make it work with all 6 providers we set up today.